# Setup

In [1]:
# Заставляем ноутбук обновлять импорты автоматически (если ты изменил код в src/)
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
from pathlib import Path
from hydra.utils import instantiate
import matplotlib.pyplot as plt

# 1. Находим корень проекта
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    print(f"Added {project_root} to sys.path")

# 2. Инициализируем Hydra и конфиги
from src.utils.notebook_setup import init_nlp_notebook # noqa: E402
cfg = init_nlp_notebook()

# 3. Инициализируем токенизатор (он нужен для DataModule и анализа)
tokenizer = instantiate(cfg.model.tokenizer).build()
print(f"Config loaded. Model: {cfg.model.architecture.model_name}")

Added c:\nlp_template to sys.path
Working directory set to: C:\nlp_template
NLP Environment ready. Config loaded: main


c:\nlp_template\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:src.core.models.tokenization:Загрузка токенизатора: DeepPavlov/rubert-base-cased
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased/4036cab694767a299f2b9e6492909664d9414229/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/resolve-cache/models/DeepPavlov/rubert-base-cased/4036cab694767a299f2b9e6492909664d9414229/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/DeepPavlov/rubert-base-cased/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Reques

Config loaded. Model: DeepPavlov/rubert-base-cased


# Data Loading

In [ ]:
from src.core.data.builder import NLPDataModule

# Используем индустриальный подход: инициализируем DataModule
datamodule = NLPDataModule(data_cfg=cfg.data, tokenizer=tokenizer)

# Метод prepare_data скачает сырые данные и применит пайплайн очистки (с кэшированием на диск)
datamodule.prepare_data()

# Метод setup загрузит готовые очищенные датасеты в память
datamodule.setup(stage="fit")

# Извлекаем тренировочный датасет для анализа
dataset = datamodule.train_dataset
print(f"Dataset schema: {dataset.column_names}")

ConfigAttributeError: Key 'dataset_loader' is not in struct
    full_key: data.dataset_loader
    object_type=dict

# Tokenization & Sequence Length Analysis

In [ ]:
# Извлекаем тексты из правильной колонки, указанной в конфиге
text_col = cfg.data.text_column
sample_texts = dataset.select(range(min(1000, len(dataset))))[text_col]

# Пример анализа длин
lengths = [len(tokenizer.encode(text)) for text in sample_texts]

plt.hist(lengths, bins=50)
plt.title("Distribution of Token Lengths (Cleaned Data)")
plt.xlabel("Token count")
plt.ylabel("Frequency")
plt.show()

# Artifact & Noise Identification

In [ ]:
import pandas as pd
import re

# Загружаем СЫРЫЕ данные напрямую для оценки изначального "шума"
raw_datasets = instantiate(cfg.data.source)
raw_train = raw_datasets["train"] if "train" in raw_datasets else raw_datasets
raw_sample_data = raw_train.select(range(min(5000, len(raw_train))))[cfg.data.text_column]

# 1. Определяем паттерны "шума" 
noise_patterns = {
    "extra_whitespace": r"\s{2,}",
    "html_tags": r"<[^>]+>",
    "non_printable": r"[^\x20-\x7E\u0400-\u04FF\n]", 
    "truncated_lines": r"^\s*\.\.\.\s*$",
}

def analyze_noise(text_list):
    stats = {}
    for name, pattern in noise_patterns.items():
        count = sum(1 for text in text_list if re.search(pattern, text))
        stats[name] = count / len(text_list)
    return stats

# Применяем на выборке сырых данных
noise_report = analyze_noise(raw_sample_data)

# Выводим отчет в виде таблицы
df_noise = pd.DataFrame.from_dict(noise_report, orient='index', columns=['percentage'])
print("--- Noise Artifacts Report (RAW DATA) ---")
print(df_noise)

# Если процент высокий — нужно проверить, что твои классы-наследники BaseCleaner это исправляют.

# Quality Audit

In [ ]:
def check_round_trip(text, tokenizer):
    tokens = tokenizer.encode(text)
    decoded = tokenizer.decode(tokens, skip_special_tokens=False)
    return tokens, decoded

# Берем сложные примеры из ОЧИЩЕННОГО датасета
clean_sample_texts = dataset.select(range(5))[cfg.data.text_column]

print(f"{'Original':<30} | {'Round-Trip (Decoded)'}")
print("-" * 80)

for text in clean_sample_texts:
    tokens, decoded = check_round_trip(text, tokenizer)
    print(f"{text[:25]+'...':<30} | {decoded[:50]+'...'}")

# ПРОВЕРКА СПЕЦ-ТОКЕНОВ
# Убедись, что твои маркеры начала/конца не превращаются в [UNK]
test_marker = "<|start_header_id|>"
print(f"\nMarker '{test_marker}' encoding: {tokenizer.encode(test_marker)}")